# Task 1: don't call a missing ball a dead ball

**Owner:** ____

**Why:** the baseline writes `D` on every frame where the tracker lost the ball (41% of graded
frames). A `D` in the middle of a possession splits it in two, and `spellF1` counts possessions,
so each fake split costs a whole possession.

**🎯 End target**
- A function `build_labels(ball, dead=None, window=...)` that turns ball positions into a full
  `H/A/D/X` label array. It must also accept Task 3's filled ball and Task 4's dead-ball mask later.
- A submission `team_task1_carry` that passes the validator, with **no `D` caused by a missing ball**.
- A possession rate in the right ballpark. The ground truth has 17 possessions in 5 min (3.4/min), mean 14.2 s.


In [ ]:
import sys; sys.path.insert(0, '..')
from pipeline.common import *
import matplotlib.pyplot as plt
import pandas as pd

trk = load_tracking()          # ball, ball_speed, players, player_speed, team -- all in metres
gt = load_public_gt()          # public ground truth, X outside the 5-minute window
graded = graded_mask()         # frames the scorer grades
half = half_of_frame()         # 1, 2, or 0 outside the graded halves
PUBLIC = (11185, 18684)        # the 5 minutes we have ground truth for (inclusive)

### 1.1 Rebuild the baseline (given)

**Do:** Run this to see the pattern every later step follows: evidence, then labels, then summarise.

**✅ Done when:** it prints 26 possessions / 10.8 s for the baseline, the same as the README table.

In [ ]:
codes = nearest_team(trk["ball"], trk["players"], trk["team"])   # HOME / AWAY / NOBODY per frame
baseline = smooth(codes, 50)
baseline[~graded] = "X"
summarise(gt, *PUBLIC)
summarise(baseline, *PUBLIC)

### 1.2 Measure the damage

**Do:** In the public window, count how many of the baseline's `D` stretches sit where the ground truth says `H` or `A`.

**Hint:** `runs(baseline[a:b+1])` gives `(label, start, end)`. Check what `gt` says over the same frames.

**✅ Done when:** you can finish the sentence "__ of the baseline's __ D-stretches in the public window are fake".

In [ ]:
# TODO

### 1.3 Carry the last team through ball gaps

**Do:** Write `carry(codes, half)`: where `codes == NOBODY`, use the last known team. Reset at the start of each half and back-fill the first few frames of a half. Never carry across halftime.

**Hint:** `pandas.Series(codes).replace(NOBODY, np.nan).groupby(half).ffill().bfill()`, or a numpy index trick with `np.maximum.accumulate`.

**✅ Done when:** inside each graded half there are zero `NOBODY` values (add an `assert`).

In [ ]:
# TODO

### 1.4 Smooth after carrying

**Do:** Run `smooth(carried, w)` for `w` in 0, 25 and 50, then `summarise` each one over `PUBLIC`.

**✅ Done when:** you have a small table of possessions/min per window. Choose the one with a plausible count, **not** the best public score (the README explains why).

In [ ]:
# TODO

### 1.5 Put in the dead frames we already know

**Do:** Set `SECOND_HALF_PREKICKOFF` to `D`, and everything outside `graded` to `X`.

**✅ Done when:** `(labels[110808:113742] == 'D').all()` and `(labels[~graded] == 'X').all()` are both True.

In [ ]:
# TODO

### 1.6 Submit and score

**Do:** Run `submit("team_task1_carry", labels)`, then `score()`, and add a row to the results log below.

**✅ Done when:** the validator says OK, and you have the poss count and spellF1 next to `baseline_smooth50`.

In [ ]:
# submit("team_task1_carry", labels)
# score()

### 1.7 Make it pluggable for Tasks 3 and 4

**Do:** Wrap 1.1–1.5 into `build_labels(ball, dead=None, window=...)`. When `dead` is given, set those frames to `D` *after* smoothing. Load the inputs with `load_stage("ball_filled")` and `load_stage("dead_ball")`. They return `None` until your teammates push their `.py` files. After a `git pull`, they rebuild automatically.

**✅ Done when:** one call builds a full submission with or without the other tasks' outputs. Once it works, move the function into `pipeline/possession.py` so it can be imported.

In [ ]:
def build_labels(ball, dead=None, window=50):
    # TODO
    ...

filled = load_stage("ball_filled")      # from Task 3 -- None until it exists
dead = load_stage("dead_ball")          # from Task 4 -- None until it exists

## Results log

One row per thing you tried. Public spellF1 is a sanity check, not a target.

| date | what changed | poss/min (public) | public spellF1 | notes |
|---|---|---|---|---|
| | baseline_smooth50 | 5.2 | 0.3256 | reference |
| | | | | |